In [14]:
!zip -r model_web.zip model_web
from google.colab import files
files.download("model_web.zip")

  adding: model_web/ (stored 0%)
  adding: model_web/weights.bin (deflated 7%)
  adding: model_web/model.json (deflated 96%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import json
import numpy as np

topology = json.loads(model.to_json())

manifest = []
parcalar = []
for w in model.weights:
    ad = w.name.split(":")[0]
    dizi = w.numpy().astype("<f4")
    manifest.append({"name": ad, "shape": list(dizi.shape), "dtype": "float32"})
    parcalar.append(dizi.tobytes())

adlar = [m["name"] for m in manifest]
print("Toplam ağırlık sayısı:", len(adlar))
print("Aynı isim tekrar ediyor mu?", len(adlar) != len(set(adlar)))
print("İlk 3 isim:", adlar[:3])
print("Son 2 isim:", adlar[-2:])

os.makedirs("model_web", exist_ok=True)
with open("model_web/weights.bin", "wb") as f:
    f.write(b"".join(parcalar))

model_json = {
    "format": "layers-model",
    "generatedBy": "keras v2",
    "convertedBy": "manual",
    "modelTopology": topology,
    "weightsManifest": [{"paths": ["weights.bin"], "weights": manifest}],
}
with open("model_web/model.json", "w") as f:
    json.dump(model_json, f)

print(os.listdir("model_web"))

Toplam ağırlık sayısı: 262
Aynı isim tekrar ediyor mu? False
İlk 3 isim: ['Conv1/kernel', 'bn_Conv1/gamma', 'bn_Conv1/beta']
Son 2 isim: ['dense/kernel', 'dense/bias']
['weights.bin', 'model.json']


In [12]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tf_keras as keras
model = keras.models.load_model("model.h5")
print("Model yüklendi")

Model yüklendi


In [11]:
model.save("model.h5")
!tensorflowjs_converter --input_format=keras model.h5 model_web
!ls model_web

/usr/local/lib/python3.13/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


/bin/bash: line 1: tensorflowjs_converter: command not found
ls: cannot access 'model_web': No such file or directory


In [10]:
from sklearn.utils.class_weight import compute_class_weight
y = train["dx"].map(cid).values
w = compute_class_weight("balanced", classes=np.arange(7), y=y)
class_weight = dict(enumerate(w))

taban = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3), include_top=False, weights="imagenet")
taban.trainable = False

model = tf.keras.Sequential([
    taban,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(7, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8,
          class_weight=class_weight)

9406464/9406464 [==============================] - 2s 0us/step
Epoch 1/8
250/250 [==============================] - 140s 522ms/step - loss: 1.8057 - accuracy: 0.6182 - val_loss: 2.8083 - val_accuracy: 0.5400
Epoch 2/8
250/250 [==============================] - 146s 573ms/step - loss: 1.9945 - accuracy: 0.6212 - val_loss: 4.7631 - val_accuracy: 0.3088
Epoch 3/8
250/250 [==============================] - 148s 580ms/step - loss: 1.9305 - accuracy: 0.6685 - val_loss: 4.5363 - val_accuracy: 0.3799
Epoch 4/8
250/250 [==============================] - 123s 479ms/step - loss: 1.8563 - accuracy: 0.6912 - val_loss: 4.5304 - val_accuracy: 0.3977
Epoch 5/8
250/250 [==============================] - 122s 475ms/step - loss: 1.5637 - accuracy: 0.7182 - val_loss: 4.0542 - val_accuracy: 0.4293
Epoch 6/8
250/250 [==============================] - 123s 480ms/step - loss: 1.7340 - accuracy: 0.6957 - val_loss: 4.0686 - val_accuracy: 0.4511
Epoch 7/8
250/250 [==============================] - 139s 544ms/ste

In [9]:
import tensorflow as tf
siniflar = sorted(meta["dx"].unique())
print(siniflar)
cid = {c: i for i, c in enumerate(siniflar)}

def yukle(yol, etiket):
    img = tf.io.read_file(yol)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (160, 160))
    img = img / 127.5 - 1.0
    return img, etiket

def cevir(img, etiket):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    return img, etiket

def hazirla(df, egitim):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values, df["dx"].map(cid).values))
    ds = ds.map(yukle, num_parallel_calls=tf.data.AUTOTUNE)
    if egitim:
        ds = ds.shuffle(1000).map(cevir)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)

train_ds = hazirla(train, True)
val_ds = hazirla(val, False)

['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [8]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, va = next(gss.split(meta, groups=meta["lesion_id"]))
train, val = meta.iloc[tr], meta.iloc[va]
print(len(train), len(val))

7991 2024


In [7]:
import glob
import pandas as pd
import numpy as np

meta = pd.read_csv("ham/HAM10000_metadata.csv")
yollar = {os.path.splitext(os.path.basename(p))[0]: p
          for p in glob.glob("ham/**/*.jpg", recursive=True)}
meta["path"] = meta["image_id"].map(yollar)
print("Eksik görüntü:", meta["path"].isna().sum())
print(meta["dx"].value_counts())

Eksik görüntü: 0
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [6]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d ham

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [04:02<00:00, 23.0MB/s]



In [5]:
from google.colab import files
files.upload()

Saving ham10000.ipynb to ham10000.ipynb


{'ham10000.ipynb': b'{"cells":[{"cell_type":"code","execution_count":1,"id":"089388bf","metadata":{"_cell_guid":"b1076dfc-b9ad-4769-8c92-a6c4dae69d19","_uuid":"8f2839f25d086af736a60e9eeb907d3b93b6e0e5","execution":{"iopub.execute_input":"2025-05-04T21:08:37.401746Z","iopub.status.busy":"2025-05-04T21:08:37.4015Z","iopub.status.idle":"2025-05-04T21:08:44.658495Z","shell.execute_reply":"2025-05-04T21:08:44.657607Z"},"papermill":{"duration":7.271698,"end_time":"2025-05-04T21:08:44.659847","exception":false,"start_time":"2025-05-04T21:08:37.388149","status":"completed"},"tags":[]},"outputs":[{"name":"stdout","output_type":"stream","text":["Requirement already satisfied: scikit-learn in /usr/local/lib/python3.11/dist-packages (1.2.2)\\r\\n","Collecting scikit-learn\\r\\n","  Downloading scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)\\r\\n","Requirement already satisfied: numpy>=1.19.5 in /usr/local/lib/python3.11/dist-packages (from scikit-lea

In [4]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
import os
for kok, klasorler, dosyalar in os.walk("/kaggle/input"):
    print(kok, len(dosyalar))
    break
print(os.listdir("/kaggle/input"))

/kaggle/input 0
[]


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
